# Adaptive Darcy macro meshes

The published sine problem of Barrenechea et al. (2026), DOI [10.1137/24M1673073](https://doi.org/10.1137/24M1673073), is evaluated using the weighted energy decomposition and an original Dörfler/red-green adaptive policy. Neither contraction nor monotonicity is assumed.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/d2b53466f416a58ccea1520e72340fa3469393c88048c549d0220510b57265d3/39_adaptive_darcy-companion.zip"
COMPANION_SHA256 = "d2b53466f416a58ccea1520e72340fa3469393c88048c549d0220510b57265d3"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/39_adaptive_darcy.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import FaceSpace, SkeletonSpace, TriangleMesh
from pymhm.fem.quadrature.material import fit_material_faces, fit_material_mesh
from pymhm.io.reservoir import CartesianCellField


## Supply equations at every adaptive level

The user callback rebuilds the same local and global primal forms on each
refined macro mesh. The analytical source is independently differentiated:

$$
\begin{aligned}
p(x,y)&=\sin(\pi x)\sin(\pi y),\\
f(x,y)&=2\pi^2\sin(\pi x)\sin(\pi y),\\
a_T(p,v)&=(\nabla p,\nabla v)_T, & L_T(v)&=(f,v)_T.
\end{aligned}
$$

`define_darcy` supplies the actual local trace blocks and global boundary form.
`refine_declared_pressure` consumes that callback through generic assembly/solve,
then reuses the energy indicator, Dörfler marking and conforming red-green
refinement kernels. This example has homogeneous Dirichlet pressure, local P2,
constant face traces and independent order-eight integration. The adaptive
policy does not imply a contraction theorem or a matched literature schedule.


In [ ]:
from examples.formulations.darcy import define_darcy
from examples.formulations.adaptive import refine_declared_pressure
from pymhm.fem.conditions import minimum_estimator_degree

macro = TriangleMesh.unit_square()
sine = lambda x: np.prod(np.sin(np.pi * x), axis=1)


def equations(mesh, skeleton):
    """Declare the same primal physical forms on each refined macro mesh."""
    return define_darcy(
        mesh,
        skeleton=skeleton,
        degree=minimum_estimator_degree(0, 2),
        quadrature_order=8,
        local_refinement=2,
        source=lambda x: 2 * np.pi**2 * sine(x),
    )


with threadpool_limits(1):
    adaptive = refine_declared_pressure(
        macro, equations, iterations=2, theta=0.5, estimator_options={"quadrature_order": 8}
    )
print(
    {
        "macro_cells": [len(s.skeleton.mesh.cells) for s in adaptive.solutions],
        "estimators": adaptive.totals.tolist(),
    }
)


## Five-level campaign

The archived campaign starts with eight macrotriangles and uses $p=\sin(2\pi x)\sin(2\pi y)$. Both $\ell=0,k=2$ and $\ell=1,k=3$ use RT2 reconstruction, four local triangles per macro, and $\theta=0.5$. The P0 trace sequence has a nonmonotone error history; all ten measured estimators exceed their independently integrated energy errors. Marked cells are blue and all field panels show the actual macro geometry.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    records = json.loads((root / "examples/results/adaptive-darcy/campaign.json").read_text())
    for row in records:
        print(row["trace_degree"], row["level"], row["macro_triangles"], row["energy_error"], row["estimator"])
    for name in ("refinement", "meshes-ell1", "fields"):
        display(Image(filename=str(root / "docs/figures/adaptive-darcy" / f"{name}.png")))